# Chapter 10 Walkthrough: Exceptions, Defensive Programming, Testing & Systematic Debugging

**Book Reference:** Chapter 10, Sections 10.1 – 10.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections.

## Section 10.3.1 — Full Exception Lifecycle: `try` / `except` / `else` / `finally`

In [ ]:
def parse_reading_with_audit(raw_text):
    """Demonstrate full try / except / else / finally lifecycle."""
    try:
        value = float(raw_text)
    except (ValueError, TypeError) as exc:
        print(f"  [EXCEPT] Rejected {raw_text!r}: {exc}")
        result = None
    else:
        print(f"  [ELSE]   Parsed clean float: {value}")
        result = value
    finally:
        print("  [FINALLY] Audit entry closed.")
    return result

r1 = parse_reading_with_audit("38.4")
r2 = parse_reading_with_audit("N/A")
print("Returned:", r1, r2)


## Section 10.3.2 — Specific Exceptions & EAFP Nested Dictionary Lookups

In [ ]:
surveillance = {
    "Dhaka": {"cases": 140, "deaths": 3},
    "Sylhet": {"cases": 25, "deaths": 0},
    "Barishal": None,
}

def get_metric(report, district, metric):
    """Fetch nested metric safely via EAFP."""
    try:
        return report[district][metric]
    except (KeyError, TypeError):
        return None

print("Dhaka cases:   ", get_metric(surveillance, "Dhaka", "cases"))
print("Sylhet deaths: ", get_metric(surveillance, "Sylhet", "deaths"))
print("Barishal cases:", get_metric(surveillance, "Barishal", "cases"))
print("Missing dist:  ", get_metric(surveillance, "Khulna", "cases"))


## Section 10.3.3 — Defensive Programming: `raise ValueError` vs. Returning `None`

In [ ]:
def compute_cfr(deaths, cases):
    """Compute Case Fatality Rate (%), raising on impossible counts."""
    if deaths < 0 or cases < 0:
        raise ValueError("deaths and cases must be non-negative")
    if cases == 0:
        return None
    return round((deaths / cases) * 100, 2)

print("Valid CFR (12/400):", compute_cfr(12, 400))
print("Zero cases (0/0):  ", compute_cfr(0, 0))


## Section 10.3.4 — Custom Domain Exception (`InvalidVitalSignError`)

In [ ]:
class InvalidVitalSignError(ValueError):
    """Raised when a clinical vital sign is outside human bounds."""

def check_temp(temp_c):
    """Validate human body temperature in [30.0, 45.0] C."""
    if isinstance(temp_c, bool) or not isinstance(temp_c, (int, float)):
        raise InvalidVitalSignError(f"Non-numeric temp: {temp_c!r}")
    if not (30.0 <= temp_c <= 45.0):
        raise InvalidVitalSignError(f"Implausible temp: {temp_c} C")
    return float(temp_c)

for sample in [37.4, 41.2, 999.0]:
    try:
        clean = check_temp(sample)
        print("Accepted:", clean)
    except InvalidVitalSignError as exc:
        print("Flagged vital sign:", exc)


## Section 10.3.5 — Automated Unit Testing (`assert` & `pytest`-Style Functions)

In [ ]:
def test_compute_cfr_nominal():
    assert compute_cfr(12, 400) == 3.0
    assert compute_cfr(0, 250) == 0.0

def test_compute_cfr_zero_cases():
    assert compute_cfr(0, 0) is None

def test_compute_cfr_rejects_negatives():
    try:
        compute_cfr(-3, 100)
    except ValueError:
        pass
    else:
        raise AssertionError("Expected ValueError for negative deaths")

for test_fn in (
    test_compute_cfr_nominal,
    test_compute_cfr_zero_cases,
    test_compute_cfr_rejects_negatives,
):
    test_fn()
    print(f"PASS: {test_fn.__name__}")


## Section 10.3.6 — Break It On Purpose! (Exception & Testing Error Anatomy)

In [ ]:
# Intentional Bug #1: ValueError from explicit defensive raise
bad_cfr = compute_cfr(-5, 100)


In [ ]:
# Intentional Bug #2: AssertionError catching a broken CFR formula
def buggy_cfr(deaths, cases):
    return round(deaths / cases, 2)   # Forgot * 100!

assert buggy_cfr(12, 400) == 3.0, "Expected 3.0%, got wrong scale!"


In [ ]:
# Intentional Bug #3: KeyError — uncaught missing nested dictionary key
report = {"Dhaka": {"cases": 140}}
dhaka_icu = report["Dhaka"]["icu_beds"]


## Section 10.4 — Worked Problem-Structuring Blueprint: Resilient Clinical Intake Pipeline

In [ ]:
def validate_intake_row(row):
    """Validate one clinic intake dict or raise InvalidVitalSignError."""
    for key in ("patient_id", "temp_c", "sbp_mmhg"):
        if key not in row:
            raise InvalidVitalSignError(f"Missing field: {key}")
    try:
        temp_c = float(row["temp_c"])
        sbp = float(row["sbp_mmhg"])
    except (ValueError, TypeError) as exc:
        raise InvalidVitalSignError(f"Non-numeric vital: {exc}")
    if not (30.0 <= temp_c <= 45.0):
        raise InvalidVitalSignError(f"Bad temp_c: {temp_c}")
    if not (50.0 <= sbp <= 260.0):
        raise InvalidVitalSignError(f"Bad sbp_mmhg: {sbp}")
    return {"patient_id": row["patient_id"], "temp_c": temp_c, "sbp": sbp}

def process_intake_batch(raw_rows):
    """Return (accepted_rows, quarantine_log) without crashing."""
    accepted = []
    quarantine = []
    for idx, row in enumerate(raw_rows):
        try:
            clean = validate_intake_row(row)
        except InvalidVitalSignError as exc:
            quarantine.append(f"Row {idx}: {exc}")
        else:
            accepted.append(clean)
    return (accepted, quarantine)

sample_rows = [
    {"patient_id": "M-01", "temp_c": "37.8", "sbp_mmhg": "118"},
    {"patient_id": "M-02", "temp_c": "999", "sbp_mmhg": "120"},
    {"patient_id": "M-03", "temp_c": "38.6"},
    {"patient_id": "M-04", "temp_c": 36.7, "sbp_mmhg": 142},
]
good, bad = process_intake_batch(sample_rows)
assert len(good) == 2 and len(bad) == 2
print("Accepted:", good)
print("Quarantined:", bad)
